# Databricks/PySpark dla Data Analyst/Engineer — Moduł 3: Transformacje i agregacje

Ten moduł rozwija DataFrame API z Modułu 2 o agregacje (`groupBy`), bibliotekę funkcji
wbudowanych `pyspark.sql.functions`, wyrażenia warunkowe (`when`/`otherwise`, odpowiednik
SQL-owego `CASE WHEN`) oraz obsługę wartości `NULL`.

## Spis treści
1. [groupBy i agregacje](#sec1)
2. [Wiele agregacji naraz](#sec2)
3. [Funkcje wbudowane: tekst i daty](#sec3)
4. [Wyrażenia warunkowe: when/otherwise](#sec4)
5. [Obsługa wartości NULL](#sec5)
6. [Podsumowanie i ćwiczenia](#sec6)


Uruchamiamy (albo pobieramy już istniejącą) `SparkSession` — punkt wejścia do wszystkiego w Sparku. W Databricks `spark` jest już gotowe automatycznie w każdym notebooku; ten sam kod działa też lokalnie.

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("kurs_spark").getOrCreate()
spark


Wczytujemy nasz wspólny zestaw danych — dane sklepu internetowego (`klienci`, `produkty`, `pracownicy`, `zamowienia`, `pozycje_zamowien`), dokładnie ten sam schemat co w kursie SQL, ale tym razem jako DataFrame'y Sparka.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(7)

# --- klienci -----------------------------------------------------------
n_klienci = 300
miasta = ["Warszawa", "Krakow", "Gdansk", "Wroclaw", "Poznan"]
segmenty = ["Nowy", "Standardowy", "Premium"]
prawdop_segmentu = [0.35, 0.45, 0.20]

klienci_pd = pd.DataFrame({
    "klient_id": np.arange(1, n_klienci + 1),
    "miasto": rng.choice(miasta, size=n_klienci, p=[0.30, 0.25, 0.15, 0.15, 0.15]),
    "segment": rng.choice(segmenty, size=n_klienci, p=prawdop_segmentu),
    "dni_od_rejestracji": rng.integers(1, 900, size=n_klienci),
})

# --- produkty ------------------------------------------------------------
n_produkty = 40
kategorie = ["Elektronika", "Odziez", "Dom", "Sport", "Ksiazki"]
srednie_ceny = {"Elektronika": 1200, "Odziez": 150, "Dom": 250, "Sport": 300, "Ksiazki": 50}

kategorie_produktow = rng.choice(kategorie, size=n_produkty)
produkty_pd = pd.DataFrame({
    "produkt_id": np.arange(1, n_produkty + 1),
    "kategoria": kategorie_produktow,
    "cena": [
        round(max(5.0, rng.normal(srednie_ceny[k], srednie_ceny[k] * 0.3)), 2)
        for k in kategorie_produktow
    ],
})

# --- pracownicy ----------------------------------------------------------
n_pracownicy = 12
regiony = ["Polnoc", "Poludnie", "Wschod", "Zachod"]
pracownicy_pd = pd.DataFrame({
    "pracownik_id": np.arange(1, n_pracownicy + 1),
    "region": rng.choice(regiony, size=n_pracownicy),
    "dzial": rng.choice(["Sprzedaz", "Obsluga"], size=n_pracownicy, p=[0.6, 0.4]),
})

# --- zamowienia ------------------------------------------------------------
n_zamowienia = 3000
klient_id_zamowien = rng.choice(klienci_pd["klient_id"], size=n_zamowienia)
pracownik_id_zamowien = rng.choice(pracownicy_pd["pracownik_id"], size=n_zamowienia)

segment_mapa = klienci_pd.set_index("klient_id")["segment"]
segment_zamowien = klient_id_zamowien.astype(object)
segment_zamowien = pd.Series(klient_id_zamowien).map(segment_mapa).values

# prawdopodobienstwo anulowania zalezy od segmentu klienta (celowo wbudowany efekt)
prawdop_anulowania = np.where(
    segment_zamowien == "Nowy", 0.22,
    np.where(segment_zamowien == "Standardowy", 0.10, 0.04),
)
anulowane = rng.binomial(1, prawdop_anulowania)
status_zamowien = np.where(
    anulowane == 1, "Anulowane",
    rng.choice(["Zlozone", "Wyslane", "Dostarczone"], size=n_zamowienia, p=[0.2, 0.3, 0.5]),
)

data_bazowa = pd.Timestamp("2024-01-01")
dni_offset = rng.integers(0, 730, size=n_zamowienia)

zamowienia_pd = pd.DataFrame({
    "zamowienie_id": np.arange(1, n_zamowienia + 1),
    "klient_id": klient_id_zamowien,
    "pracownik_id": pracownik_id_zamowien,
    "data_zamowienia": [data_bazowa + pd.Timedelta(days=int(d)) for d in dni_offset],
    "status": status_zamowien,
})

# --- pozycje_zamowien ------------------------------------------------------
wiersze_pozycji = []
pozycja_id = 1
cena_produktu = produkty_pd.set_index("produkt_id")["cena"]

for zamowienie_id in zamowienia_pd["zamowienie_id"]:
    liczba_pozycji = rng.integers(1, 5)
    wybrane_produkty = rng.choice(produkty_pd["produkt_id"], size=liczba_pozycji, replace=False)
    for produkt_id in wybrane_produkty:
        ilosc = int(rng.integers(1, 4))
        rabat = rng.choice([0.0, 0.05, 0.10, 0.15], p=[0.55, 0.20, 0.15, 0.10])
        wiersze_pozycji.append({
            "pozycja_id": pozycja_id,
            "zamowienie_id": int(zamowienie_id),
            "produkt_id": int(produkt_id),
            "ilosc": ilosc,
            "cena_jednostkowa": float(cena_produktu[produkt_id]),
            "rabat": float(rabat),
        })
        pozycja_id += 1

pozycje_zamowien_pd = pd.DataFrame(wiersze_pozycji)

# --- konwersja na DataFrame'y Sparka ---------------------------------------
klienci = spark.createDataFrame(klienci_pd)
produkty = spark.createDataFrame(produkty_pd)
pracownicy = spark.createDataFrame(pracownicy_pd)
zamowienia = spark.createDataFrame(zamowienia_pd)
pozycje_zamowien = spark.createDataFrame(pozycje_zamowien_pd)

klienci.show(5)


<a id="sec1"></a>
## 1. groupBy i agregacje

`groupBy()` + `agg()` (albo skrótowe metody typu `.count()`) odpowiadają `GROUP BY` z
SQL-a — dokładnie to, czym zajmowaliśmy się w Module 4 kursu SQL, teraz na Sparku.

In [ ]:
klienci.groupBy("segment").count().show()


In [ ]:
from pyspark.sql import functions as F

produkty.groupBy("kategoria").agg(F.avg("cena").alias("srednia_cena")).show()


> ⚡ **groupBy() sam w sobie nic nie liczy**
>
> Podobnie jak inne transformacje z Modułu 1 — `df.groupBy("x")` samo w sobie zwraca obiekt `GroupedData`, jeszcze nie wynik. Dopiero doklejenie `.agg(...)` (albo `.count()`) i wywołanie akcji (`.show()`) faktycznie uruchamia obliczenia.

<a id="sec2"></a>
## 2. Wiele agregacji naraz

`agg()` przyjmuje wiele funkcji agregujących naraz — dokładnie jak wiele kolumn po
`SELECT` w zapytaniu z `GROUP BY`.

In [ ]:
produkty.groupBy("kategoria").agg(
    F.count("*").alias("liczba_produktow"),
    F.avg("cena").alias("srednia_cena"),
    F.min("cena").alias("cena_min"),
    F.max("cena").alias("cena_max"),
    F.round(F.stddev("cena"), 2).alias("odchylenie_std"),
).orderBy(F.col("srednia_cena").desc()).show()


Możemy też agregować po WIELU kolumnach jednocześnie — Spark grupuje po każdej unikalnej
KOMBINACJI wartości tych kolumn, dokładnie jak `GROUP BY kolumna1, kolumna2` w SQL-u.

In [ ]:
zamowienia_z_klientami = zamowienia.join(klienci, "klient_id")

zamowienia_z_klientami.groupBy("segment", "status").agg(
    F.count("*").alias("liczba_zamowien")
).orderBy("segment", "status").show(20)


<a id="sec3"></a>
## 3. Funkcje wbudowane: tekst i daty

`pyspark.sql.functions` (zwykle importowane jako `F`) to ogromna biblioteka gotowych
funkcji — string, matematyczne, daty, agregujące, okienkowe (Moduł 7) i inne. Zamiast
pisać własną logikę w Pythonie (co byłoby wolne — zobacz Moduł 8 o UDF-ach), niemal
zawsze warto najpierw sprawdzić, czy potrzebna funkcja już tam jest.

In [ ]:
produkty.select(
    "kategoria",
    F.upper("kategoria").alias("kategoria_wielkimi"),
    F.length("kategoria").alias("dlugosc_nazwy"),
).show(5)


In [ ]:
zamowienia.select(
    "data_zamowienia",
    F.year("data_zamowienia").alias("rok"),
    F.month("data_zamowienia").alias("miesiac"),
    F.date_add("data_zamowienia", 30).alias("data_plus_30_dni"),
).show(5)


> ⚡ **Ściągawka: gdzie szukać funkcji**
>
> Pełna lista jest w oficjalnej dokumentacji `pyspark.sql.functions` — warto ją mieć pod ręką. Najczęściej używane grupy: tekstowe (`upper`, `lower`, `trim`, `substring`, `concat`, `regexp_replace`), daty (`year`, `month`, `datediff`, `date_add`), matematyczne (`round`, `abs`, `sqrt`) i warunkowe (`when`, `coalesce`, `isnull`).

<a id="sec4"></a>
## 4. Wyrażenia warunkowe: when/otherwise

`F.when(warunek, wartosc).when(...).otherwise(domyslna)` to odpowiednik SQL-owego `CASE
WHEN ... THEN ... ELSE ... END` — budujemy nową kolumnę na podstawie serii warunków
sprawdzanych po kolei.

In [ ]:
klienci_z_typem = klienci.withColumn(
    "typ_klienta",
    F.when(F.col("segment") == "Premium", "VIP")
    .when(F.col("segment") == "Nowy", "Nowicjusz")
    .otherwise("Zwykly"),
)
klienci_z_typem.groupBy("typ_klienta").count().show()


> ⚠️ **Kolejność warunków w when() ma znaczenie**
>
> Podobnie jak w `CASE WHEN` z SQL-a, Spark sprawdza warunki PO KOLEI i bierze PIERWSZY pasujący — jeśli dwa warunki mogłyby pasować do tego samego wiersza, wygrywa ten wcześniejszy w łańcuchu `.when()`.

<a id="sec5"></a>
## 5. Obsługa wartości NULL

Nasze dane akurat nie mają braków, ale w praktyce prawie zawsze się z nimi spotkasz.
Kluczowe narzędzia: `isNull()`/`isNotNull()` (filtrowanie), `fillna()`/`na.fill()`
(wypełnianie), `dropna()`/`na.drop()` (usuwanie wierszy z brakami), `F.coalesce()`
(pierwsza niepusta wartość z kilku kolumn).

In [ ]:
# Male dane testowe z brakami, zeby pokazac mechanike
dane_testowe = spark.createDataFrame(
    [(1, "Warszawa"), (2, None), (3, "Gdansk")], ["id", "miasto"]
)
dane_testowe.filter(F.col("miasto").isNull()).show()


In [ ]:
dane_testowe.fillna({"miasto": "Nieznane"}).show()


In [ ]:
dane_testowe.na.drop().show()


> ⚠️ **== None nie zadziała tak, jak w Pythonie**
>
> `F.col("miasto") == None` NIE wykryje braków (w logice SQL/Sparka porównanie z `NULL` zawsze daje `NULL`, nigdy `True`). Do sprawdzania braków ZAWSZE używaj `.isNull()` / `.isNotNull()`, nigdy `== None`.

<a id="sec6"></a>
## 6. Podsumowanie i ćwiczenia

W tym module poznaliśmy:
- `groupBy()` + `agg()` z wieloma funkcjami agregującymi naraz,
- grupowanie po wielu kolumnach jednocześnie,
- bibliotekę funkcji wbudowanych `pyspark.sql.functions` (tekst, daty),
- wyrażenia warunkowe `when()`/`otherwise()` (odpowiednik `CASE WHEN`),
- obsługę wartości `NULL`: `isNull`/`isNotNull`, `fillna`, `dropna`, `coalesce`.

### 📝 Ćwiczenie 1: Średnia wartość zamówienia per miasto

Połącz `zamowienia` z `klienci`, pogrupuj po `miasto` i policz liczbę zamówień w każdym mieście.

<details>
<summary>Pokaż rozwiązanie</summary>

```python
zamowienia.join(klienci, "klient_id").groupBy("miasto").agg(
    F.count("*").alias("liczba_zamowien")
).orderBy(F.col("liczba_zamowien").desc()).show()
```

</details>

### 📝 Ćwiczenie 2: Kategoryzacja produktów po cenie

Dodaj do `produkty` kolumnę `przedzial_cenowy` przez `when/otherwise`: `'Tani'` dla ceny poniżej 100, `'Sredni'` dla 100-500, `'Drogi'` powyżej 500.

<details>
<summary>Pokaż rozwiązanie</summary>

```python
produkty.withColumn(
    "przedzial_cenowy",
    F.when(F.col("cena") < 100, "Tani")
    .when(F.col("cena") <= 500, "Sredni")
    .otherwise("Drogi"),
).groupBy("przedzial_cenowy").count().show()
```

</details>

### 📝 Ćwiczenie 3: Miesiąc z największą liczbą zamówień

Wyciągnij miesiąc (`F.month`) z `data_zamowienia`, pogrupuj po nim i znajdź miesiąc z największą liczbą zamówień.

<details>
<summary>Pokaż rozwiązanie</summary>

```python
zamowienia.withColumn("miesiac", F.month("data_zamowienia")).groupBy("miesiac").agg(
    F.count("*").alias("liczba_zamowien")
).orderBy(F.col("liczba_zamowien").desc()).show(3)
```

</details>

### 📝 Ćwiczenie 4: Uzupełnianie braków

Dla `dane_testowe` z sekcji 5, wypełnij braki w `miasto` wartością `'Brak danych'` i wypisz wynik.

<details>
<summary>Pokaż rozwiązanie</summary>

```python
dane_testowe.fillna({"miasto": "Brak danych"}).show()
```

</details>

> 🔥 **Wyzwanie: pełna analiza wartości zamówień per segment i status**
>
> Połącz `zamowienia`, `klienci` i `pozycje_zamowien`, dodaj kolumnę `wartosc_pozycji` (jak w Module 2), zagreguj SUMĘ wartości per `(segment, status)`, i dodaj kolumnę `kategoria_wartosci` przez `when/otherwise`: `'Wysoka'` gdy suma > 5000, w przeciwnym razie `'Standardowa'`.

<details>
<summary>Pokaż rozwiązanie</summary>

```python
polaczone = (
    zamowienia.join(klienci, "klient_id")
    .join(pozycje_zamowien, "zamowienie_id")
    .withColumn("wartosc_pozycji", F.col("ilosc") * F.col("cena_jednostkowa") * (1 - F.col("rabat")))
)

wynik = polaczone.groupBy("segment", "status").agg(
    F.round(F.sum("wartosc_pozycji"), 2).alias("suma_wartosci")
).withColumn(
    "kategoria_wartosci",
    F.when(F.col("suma_wartosci") > 5000, "Wysoka").otherwise("Standardowa"),
)
wynik.orderBy(F.col("suma_wartosci").desc()).show(20)
```

</details>

## Co dalej?

W Module 3 łączyliśmy tabele "na skróty", zanim formalnie omówiliśmy `join`. **Moduł 4**
nadrabia to porządnie: typy joinów (inner/left/right/outer/semi/anti), broadcast join i
pierwsze spojrzenie na to, jak kosztowne bywa łączenie dużych tabel.